In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

In [20]:
ticker = "AAPL"
start_date = "2010-01-01"

data = yf.download(ticker, start=start_date,auto_adjust=False)
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)
data.columns.name = None
data.index.name = "Date"  # optional, makes the index column named when saving
data_dir = Path("/content/drive/MyDrive/datascience/apple")
data_dir.mkdir(parents=True, exist_ok=True)
data.to_csv(data_dir / "AAPL_raw.csv") # saves from having to load again

[*********************100%***********************]  1 of 1 completed


data checks

In [21]:
data.head()


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2010-01-04,6.406478,7.643214,7.660714,7.585000,7.622500,493729600
2010-01-05,6.417556,7.656429,7.699643,7.616071,7.664286,601904800
2010-01-06,6.315477,7.534643,7.686786,7.526786,7.656429,552160000
2010-01-07,6.303802,7.520714,7.571429,7.466071,7.562500,477131200
2010-01-08,6.345712,7.570714,7.571429,7.466429,7.510714,447610800


In [17]:
data.tail()

,Close,High,Low,Open,Volume
Date,,,,,
2026-06-17,295.950012,302.070007,294.359985,300.850006,42745100
2026-06-18,298.010010,300.570007,295.619995,298.109985,85962200
2026-06-22,297.010010,302.420013,296.760010,297.309998,44879900
2026-06-23,294.299988,301.640015,294.179993,297.540009,52010900
2026-06-24,293.079987,299.700012,292.940002,295.359985,52882100


In [23]:
rows, cols = data.shape
print(f"Rows: {rows}, Columns: {cols}")
data.info()
data.index.is_monotonic_increasing

Rows: 4143, Columns: 6
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4143 entries, 2010-01-04 to 2026-06-24
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  4143 non-null   float64
 1   Close      4143 non-null   float64
 2   High       4143 non-null   float64
 3   Low        4143 non-null   float64
 4   Open       4143 non-null   float64
 5   Volume     4143 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 226.6 KB


True

In [24]:
data.isna().sum()

,0
Adj Close,0
Close,0
High,0
Low,0
Open,0
Volume,0


In [28]:
first_date  = data.index.min()
last_date   = data.index.max()
n_records   = len(data)
any_missing = data.isna().any().any()

print("First date:", first_date)
print("Latest date:", last_date)
print("Number of trading-day records:", n_records)
print("Any missing values:", any_missing)

# Example difference between opening and closing prices (can also take mean)
data["Close_minus_Open"] = data["Close"] - data["Open"]
print(data["Close_minus_Open"].head())
print("Mean difference between close and open prices",data['Close_minus_Open'].mean())

First date: 2010-01-04 00:00:00
Latest date: 2026-06-24 00:00:00
Number of trading-day records: 4143
Any missing values: False
Date
2010-01-04    0.020714
2010-01-05   -0.007857
2010-01-06   -0.121786
2010-01-07   -0.041786
2010-01-08    0.060000
Name: Close_minus_Open, dtype: float64
Mean difference between close and open prices 0.06308354567192385


# Exploratory Data Analysis

In [30]:
# 1. Daily percentage return (using close prices)
data["Daily_Return"] = data["Close"].pct_change()

# 2. Price and volume summary statistics
min_close    = data["Close"].min()
max_close    = data["Close"].max()
avg_close    = data["Close"].mean()
median_close = data["Close"].median()
avg_volume   = data["Volume"].mean()
avg_daily_ret = data["Daily_Return"].mean()

print("Minimum closing price:", min_close)
print("Maximum closing price:", max_close)
print("Average closing price:", avg_close)
print("Median closing price:", median_close)
print("Average daily trading volume:", avg_volume)
print("Average daily return:", avg_daily_ret)

Minimum closing price: 6.85892915725708
Maximum closing price: 315.20001220703125
Average closing price: 85.40084701098884
Median closing price: 42.877498626708984
Average daily trading volume: 214090527.78180063
Average daily return: 0.00103701614872089


In [31]:
# Absolute daily moves
data["Abs_Daily_Return"] = data["Daily_Return"].abs()

# Top 1% largest absolute returns
threshold_ret = data["Abs_Daily_Return"].quantile(0.99)
big_moves = data[data["Abs_Daily_Return"] >= threshold_ret]

# Top 1% largest volumes
threshold_vol = data["Volume"].quantile(0.99)
big_volume_days = data[data["Volume"] >= threshold_vol]

print("Number of very large move days (top 1%):", len(big_moves))
print("Number of very high volume days (top 1%):", len(big_volume_days))

Number of very large move days (top 1%): 42
Number of very high volume days (top 1%): 42


In [35]:
big_moves[["Close", "Daily_Return", "Volume"]]


,Close,Daily_Return,Volume
Date,,,
2010-04-21,9.257857,0.059814,982391200
2010-05-10,9.071071,0.076868,984306400
2012-01-25,15.952143,0.062439,958314000
2012-04-25,21.785713,0.088741,905777600
2012-11-19,20.204643,0.072108,823317600
2012-12-05,19.242500,-0.064357,1044638000
2013-01-24,16.089287,-0.123558,1460852400
2014-01-28,18.089287,-0.079927,1065523200
2014-04-24,20.277500,0.081982,759911600


In [36]:
big_volume_days[["Close", "Daily_Return", "Volume"]]

,Close,Daily_Return,Volume
Date,,,
2010-01-25,7.252500,0.026903,1065699600
2010-01-26,7.355000,0.014133,1867110000
2010-01-27,7.424286,0.009420,1722568400
2010-01-28,7.117500,-0.041322,1173502400
2010-01-29,6.859286,-0.036279,1245952400
2010-05-06,8.794643,-0.038048,1285860800
2010-05-07,8.423571,-0.042193,1676018400
2010-05-19,8.869286,-0.015930,1025726800
2010-05-20,8.491429,-0.042603,1282915200


Apple’s closing price in this sample ranges from about 6.86 at the low end to about 315.20 at the high end, showing how much the stock has grown over the period. The average closing price is roughly 85.40, while the median is around 42.88, which suggests that many of the earlier observations are clustered at lower price levels and later years bring much higher prices.

Average daily trading volume is about 214 million shares, indicating very high liquidity and constant investor activity in Apple’s stock. The average daily return is approximately 0.10%, so on a typical trading day the price tends to move slightly upward, though the effect is small at the daily scale.

There are 42 days in the top 1% of absolute daily returns and 42 days in the top 1% of trading volume. This shows that a relatively small number of days experience unusually large price moves and unusually heavy trading, which likely correspond to major events such as earnings announcements.

It is also worth noting that big‑move days are spread fairly evenly across the years, which means large percentage price changes occur throughout the sample rather than being concentrated in one early “high‑risk” period. In contrast, unusually high‑volume days appear only up to early 2014, suggesting that trading activity later became more stable or structurally different, so volatility in recent years is less often accompanied by extreme volume spikes.